Phase 5: GNN edge classifier (GINE in plain PyTorch), trained on Kaggle and logged to DagsHub.

Every transaction is an edge between two accounts. The model scores each transaction from the graph of all transactions on
EARLIER DAYS (a daily snapshot) plus the transaction's own features. Nothing from its own day or later is visible. Day-1
transactions are not trained on (their snapshot is empty).

What gets stored on DagsHub (experiment "papertrail"): one run per seed (gnn_gine_seed0/1/2) with params, per-epoch curves, final
metrics and the model weights (.pt); and a summary run (gnn_gine) with mean +- std over seeds, a 3-seed ensemble, and the scores
file (gnn_scores.parquet) that the blend notebook reads.

How to run:
  1. DEBUG = True (cell 2), accelerator None: a 5% sample, one seed, a few epochs. Takes a few minutes. All checks must pass.
  2. DEBUG = False, accelerator None (CPU is enough: the neighbour sampling is CPU-bound, so a GPU barely helps),
     Internet ON, DAGSHUB_TOKEN secret attached, input papertrail-processed. Save Version -> Save & Run All (Commit).
     Expect roughly 30 to 60 minutes for 3 seeds.

Reference to beat (test split, notebook 03d, 3 seeds): XGBoost with history features PR-AUC 0.442 +- 0.004, F1 0.499.



In [ ]:
!pip install -q git+https://github.com/abhibarnwal707/PaperTrail.git
import os
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
import dagshub, mlflow
dagshub.init(repo_owner="abhibarnwal707", repo_name="papertrail", mlflow=True)


In [ ]:
import time as _t; print("[cell 2: configuration] started", _t.strftime("%H:%M:%S"))
import glob
import json
import os
from dataclasses import replace

import numpy as np
import polars as pl
import torch

from papertrail.gnn.data import TEST, TRAIN, VAL, build_edge_table, make_batch
from papertrail.gnn.train import GNNConfig, SnapshotCache, overfit_check, predict, train_gnn
from papertrail.metrics import best_f1_threshold, evaluate
from papertrail.split import assert_valid_split, split_summary, temporal_split

DEBUG = True            # True: 5% sample, one seed, a few epochs.  False: full data, 3 seeds, full training.
SUFFIX = "_debug" if DEBUG else ""
SEEDS = (0,) if DEBUG else (0, 1, 2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_cfg = GNNConfig(hidden=32, batch_size=256, steps_per_epoch=40, max_epochs=3, patience=3, eval_batch=1024) if DEBUG else GNNConfig()
print("device:", device, "| DEBUG =", DEBUG, "| seeds:", SEEDS)
print(base_cfg)


In [ ]:
import time as _t; print("[cell 3: load and split] started", _t.strftime("%H:%M:%S"))
DATA = "/kaggle/input/datasets/abhibarnwal707/papertrail-processed/transactions.parquet"
if not os.path.exists(DATA):
    DATA = glob.glob("/kaggle/input/**/transactions.parquet", recursive=True)[0]
df = pl.read_parquet(DATA).sort(["ts", "txn_id"])
if DEBUG:
    df = df.sample(fraction=0.05, seed=0).sort(["ts", "txn_id"])        # random 5%: the structure is thin, only the plumbing matters
summary = split_summary(df)
print(summary)
assert_valid_split(summary)
train, val, test = temporal_split(df)

t0 = _t.time()
table = build_edge_table(train, val, test)
all_df = pl.concat([train, val, test])
print(f"edge table in {_t.time() - t0:.0f}s: {table.n_edges} edges, {table.n_nodes} accounts, "
      f"{len(table.feature_names)} edge features, positives {int(table.label.sum())}")
print("days:", {int(d): int((table.day == d).sum()) for d in np.unique(table.day)})


In [ ]:
import time as _t; print("[cell 4: leakage check] started", _t.strftime("%H:%M:%S"))
snaps = SnapshotCache(table)
rng = np.random.default_rng(0)
for split, name in ((TRAIN, "train"), (VAL, "val"), (TEST, "test")):
    ids = table.ids(split, min_day=2)
    day = int(table.day[ids].min())
    snap = snaps[day]
    on_day = ids[table.day[ids] == day]
    sub = rng.choice(on_day, size=min(512, len(on_day)), replace=False)
    b = make_batch(table, snap, sub, base_cfg.fanouts, rng)
    used = b["edge_global"] % max(snap.n, 1)                             # which transaction each sampled edge is
    assert (table.day[used] < day).all(), "a sampled neighbour is not from an earlier day"
    assert not np.isin(sub, used).any(), "a target edge appears among its own neighbours"
    print(f"{name:5s} day {day}: snapshot has {snap.n} transactions; batch of {len(sub)} targets -> "
          f"{len(b['nodes'])} nodes, {b['edge_index'].shape[1]} edges; latest neighbour day {int(table.day[used].max()) if len(used) else '-'}")
print("no-future-edge check passed")


In [ ]:
import time as _t; print("[cell 5: overfit check] started", _t.strftime("%H:%M:%S"))
final_loss = overfit_check(table, base_cfg, device, n=256, steps=300)
assert final_loss < 0.1, f"could not overfit a tiny batch (final loss {final_loss:.3f}); the pipeline has a bug"
print("overfit check passed")


In [ ]:
import time as _t; print("[cell 6: train] started", _t.strftime("%H:%M:%S"))
val_ids, test_ids = table.ids(VAL), table.ids(TEST)
y_val, y_test = table.label[val_ids], table.label[test_ids]
test_ach = (all_df[test_ids]["pay_format"] == "ACH").to_numpy()


def evaluate_scores(p_val, p_test):
    """Same protocol as every other model: threshold from validation, test scored once, plus the ACH-only slice."""
    thr = best_f1_threshold(y_val, p_val)
    res = {"threshold": float(thr)}
    for k, v in evaluate(y_val, p_val, thr).items():
        res[f"val_{k}"] = v
    for k, v in evaluate(y_test, p_test, thr).items():
        res[f"test_{k}"] = v
    for k, v in evaluate(y_test[test_ach], p_test[test_ach], thr).items():
        res[f"test_ach_{k}"] = v
    return res


results, scores, histories = {}, {}, {}
for seed in SEEDS:
    cfg = replace(base_cfg, seed=seed)
    t0 = _t.time()
    with mlflow.start_run(run_name=f"gnn_gine_seed{seed}{SUFFIX}"):
        mlflow.log_params({**cfg.as_dict(), "debug": DEBUG, "n_edges": table.n_edges, "n_edge_features": len(table.feature_names)})
        model, history, snaps = train_gnn(table, cfg, device, val_ids, snaps=snaps)
        for h in history:
            mlflow.log_metrics({"epoch_train_loss": h["train_loss"], "epoch_val_pr_auc": h["val_pr_auc"]}, step=h["epoch"])
        p_val = predict(model, table, val_ids, cfg, device, snaps)
        p_test = predict(model, table, test_ids, cfg, device, snaps)
        res = evaluate_scores(p_val, p_test)
        mlflow.log_metrics(res)
        mlflow.log_param("epochs_run", len(history))
        weights = f"/kaggle/working/gnn_gine_seed{seed}{SUFFIX}.pt"
        torch.save(model.state_dict(), weights)
        mlflow.log_artifact(weights)
    results[seed], scores[seed], histories[seed] = res, (p_val, p_test), history
    print(f"seed {seed}: {len(history)} epochs, {_t.time() - t0:.0f}s | val PR-AUC {res['val_pr_auc']:.3f} | test PR-AUC {res['test_pr_auc']:.3f} "
          f"F1 {res['test_f1']:.3f} recall@1%FPR {res['test_recall_at_1pct_fpr']:.3f} precision@1000 {res['test_precision_at_1000']:.3f}")


In [ ]:
import time as _t; print("[cell 7: summary] started", _t.strftime("%H:%M:%S"))
KEYS = ["val_pr_auc", "test_pr_auc", "test_f1", "test_recall_at_1pct_fpr", "test_precision_at_1000", "test_ach_pr_auc", "test_ach_f1"]
summary_rows = {k: (float(np.mean([r[k] for r in results.values()])), float(np.std([r[k] for r in results.values()]))) for k in KEYS}
for k, (m, s) in summary_rows.items():
    print(f"{k:28s} {m:.4f} +- {s:.4f}")

ens_val = np.mean([scores[s][0] for s in SEEDS], axis=0)
ens_test = np.mean([scores[s][1] for s in SEEDS], axis=0)
ens = evaluate_scores(ens_val, ens_test)
print(f"\nensemble of {len(SEEDS)} seeds (average probability): test PR-AUC {ens['test_pr_auc']:.3f}  F1 {ens['test_f1']:.3f}  "
      f"recall@1%FPR {ens['test_recall_at_1pct_fpr']:.3f}  precision@1000 {ens['test_precision_at_1000']:.3f}")
print("reference (03d, XGBoost history features, 3 seeds): test PR-AUC 0.442 +- 0.004, F1 0.499")

scores_df = pl.DataFrame({
    "txn_id": np.concatenate([all_df["txn_id"].to_numpy()[val_ids], all_df["txn_id"].to_numpy()[test_ids]]),
    "split": ["val"] * len(val_ids) + ["test"] * len(test_ids),
    "label": np.concatenate([y_val, y_test]).astype(int),
    **{f"gnn_s{s}": np.concatenate(scores[s]) for s in SEEDS},
    "gnn_mean": np.concatenate([ens_val, ens_test]),
})
scores_path = f"/kaggle/working/gnn_scores{SUFFIX}.parquet"
scores_df.write_parquet(scores_path)
summary_path = f"/kaggle/working/gnn_summary{SUFFIX}.json"
with open(summary_path, "w") as fh:
    json.dump({"config": base_cfg.as_dict(), "seeds": list(SEEDS), "per_seed": {str(s): results[s] for s in SEEDS},
               "mean_std": summary_rows, "ensemble": ens, "history": {str(s): histories[s] for s in SEEDS}, "debug": DEBUG}, fh, indent=2)

with mlflow.start_run(run_name=f"gnn_gine{SUFFIX}"):
    mlflow.log_params({**base_cfg.as_dict(), "seeds": len(SEEDS), "debug": DEBUG})
    mlflow.log_metrics({f"{k}_{stat}": v[i] for k, v in summary_rows.items() for i, stat in enumerate(("mean", "std"))})
    mlflow.log_metrics({f"ensemble_{k}": v for k, v in ens.items() if k.startswith(("val_pr", "test_"))})
    mlflow.log_artifact(scores_path)
    mlflow.log_artifact(summary_path)
print("Saved", scores_path, "and", summary_path, "(also logged to DagsHub). Download gnn_summary.json into PaperTrail/reports/.")
